# Particle I/O and Catalogue Conversion

This notebook follows one catalogue through the complete PyHermes input boundary: download the original Quijote FoF files, read them with the native FoF reader, convert the halo catalogue to NPZ and BIN, and read both converted files back through the shared particle interface.

Start with `quick_start.ipynb` for the shortest complete calculation. Return here when adapting a catalogue of your own.

## What this notebook emphasizes

- downloading and extracting the original FoF archive
- reading a Quijote/Pylians FoF directory
- constructing the NPZ catalogue distributed for the public Quick Start
- constructing and reading an equivalent raw BIN table
- converging different source formats on one PyHermes particle dictionary


In [1]:
from pathlib import Path
import os

import numpy as np

from pyhermes.io import read_particle_data

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "examples":
    examples_dir = cwd.parent
elif cwd.name == "examples":
    examples_dir = cwd
elif (cwd / "examples").is_dir():
    examples_dir = cwd / "examples"
else:
    raise RuntimeError("Run this notebook from the project root, examples/, or examples/notebooks/.")

os.chdir(examples_dir)
data_dir = Path("./data")
print(f"Working directory: {Path.cwd()}")


Working directory: /Users/xutengpeng/xutp/PycharmProjects/PyHermes/examples


## 1. Download and read the original FoF catalogue

The shell commands below download `quijote_halos.tar.gz` and extract the original `group_tab` data directly under `./data`. macOS metadata files are excluded during extraction. The NPZ and BIN transformations remain fully visible in the following sections.


In [2]:
!mkdir -p ./data
!curl --fail -L \
  -A 'Mozilla/5.0' \
  -e 'https://pyhermes.astroslacker.com/' \
  -o ./data/quijote_halos.tar.gz \
  'https://pyhermes.astroslacker.com/downloads/quijote_halos.tar.gz'

!tar -xzf ./data/quijote_halos.tar.gz -C ./data --exclude='._*' --exclude='__MACOSX'


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 9554k  100 9554k    0     0  4753k      0  0:00:02  0:00:02 --:--:-- 4753k


In [3]:
fof_root = data_dir / "quijote_halos" / "8000"
groups_dir = fof_root / "groups_004"

halo_data = read_particle_data(
    fof_root,
    data_format="fof",
    snapnum=4,
    redshift=0.0,
    fields={
        "vx": "vel_x",
        "vy": "vel_y",
        "vz": "vel_z",
        "mass": "mass",
        "npart": "npart",
    },
)

print(f"haloes: {halo_data['size']:,}")
print(f"positions: {halo_data['pos'].shape} {halo_data['pos'].dtype}")
print(f"fields: {sorted(key for key in halo_data if key not in {'pos', 'size'})}")


00:44:30 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: fof
00:44:30 - INFO - pyhermes.io.readers:read_fof - Reading Quijote FoF halo data from ---> data/quijote_halos/8000 <---
haloes: 406,728
positions: (406728, 3) float32
fields: ['mass', 'npart', 'vx', 'vy', 'vz']


## 2. Construct and read the public NPZ catalogue

The public Quick Start file at `https://pyhermes.astroslacker.com/downloads/quijote_halos_8000_snap004.npz` uses the array contract shown directly in the next cell:

- `pos`: comoving position in $h^{-1}\mathrm{Mpc}$
- `vel_x`, `vel_y`, `vel_z`: peculiar velocity components in $\mathrm{km}\,\mathrm{s}^{-1}$
- `mass`: halo mass in $h^{-1}M_\odot$
- `npart`: number of simulation particles in the FoF group

All arrays are stored as `float32`. The locally reconstructed file is placed beside the source `group_tab` file so the provenance and conversion products stay together.


In [4]:
npz_path = groups_dir / "quijote_halos_8000_snap004.npz"
np.savez_compressed(
    npz_path,
    pos=np.ascontiguousarray(halo_data["pos"], dtype=np.float32),
    vel_x=np.ascontiguousarray(halo_data["vx"], dtype=np.float32),
    vel_y=np.ascontiguousarray(halo_data["vy"], dtype=np.float32),
    vel_z=np.ascontiguousarray(halo_data["vz"], dtype=np.float32),
    mass=np.ascontiguousarray(halo_data["mass"], dtype=np.float32),
    npart=np.ascontiguousarray(halo_data["npart"], dtype=np.float32),
)
print(f"Wrote {npz_path} ({npz_path.stat().st_size / 2**20:.2f} MiB)")


Wrote data/quijote_halos/8000/groups_004/quijote_halos_8000_snap004.npz (8.55 MiB)


In [5]:
npz_fields = {
    "vx": "vel_x",
    "vy": "vel_y",
    "vz": "vel_z",
    "mass": "mass",
    "npart": "npart",
}
npz_data = read_particle_data(
    npz_path,
    data_format="npz",
    fields=npz_fields,
)

for name in ("pos", "vx", "vy", "vz", "mass", "npart"):
    np.testing.assert_allclose(npz_data[name], halo_data[name])

units = {
    "pos": "Mpc/h",
    "vel_x": "km/s",
    "vel_y": "km/s",
    "vel_z": "km/s",
    "mass": "Msun/h",
    "npart": "particle count",
}
with np.load(npz_path) as archive:
    for name in archive.files:
        values = archive[name]
        print(f"{name:>6s}: shape={values.shape}, dtype={values.dtype}, unit={units[name]}")


00:44:34 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: npz
00:44:34 - INFO - pyhermes.io.readers:read_npz - Reading particle data from ---> data/quijote_halos/8000/groups_004/quijote_halos_8000_snap004.npz <---
   pos: shape=(406728, 3), dtype=float32, unit=Mpc/h
 vel_x: shape=(406728,), dtype=float32, unit=km/s
 vel_y: shape=(406728,), dtype=float32, unit=km/s
 vel_z: shape=(406728,), dtype=float32, unit=km/s
  mass: shape=(406728,), dtype=float32, unit=Msun/h
 npart: shape=(406728,), dtype=float32, unit=particle count


## 3. Construct and read the equivalent BIN catalogue

A raw BIN file stores values but no names, shape, dtype, or units. We therefore define its eight-column layout beside both the writer and reader: `(x, y, z, vx, vy, vz, mass, npart)`. Keeping this mapping visible is the binary format contract.


In [6]:
bin_path = groups_dir / "quijote_halos_8000_snap004.bin"
bin_table = np.column_stack(
    [
        halo_data["pos"],
        halo_data["vx"],
        halo_data["vy"],
        halo_data["vz"],
        halo_data["mass"],
        halo_data["npart"],
    ]
).astype(np.float32, copy=False)
bin_table.tofile(bin_path)
print(f"Wrote {bin_path} with shape {bin_table.shape}")


Wrote data/quijote_halos/8000/groups_004/quijote_halos_8000_snap004.bin with shape (406728, 8)


In [7]:
bin_data = read_particle_data(
    bin_path,
    data_format="bin",
    dtype="float32",
    ncols=8,
    pos_cols=[0, 1, 2],
    fields={
        "vx": 3,
        "vy": 4,
        "vz": 5,
        "mass": 6,
        "npart": 7,
    },
)

for name in ("pos", "vx", "vy", "vz", "mass", "npart"):
    np.testing.assert_allclose(bin_data[name], halo_data[name])
print(f"Reloaded {bin_data['size']:,} haloes from the BIN table")


00:44:37 - INFO - pyhermes.io.readers:read_particle_data - Selected input particle format: bin
00:44:37 - INFO - pyhermes.io.readers:read_bin - Reading particle data from ---> data/quijote_halos/8000/groups_004/quijote_halos_8000_snap004.bin <---
Reloaded 406,728 haloes from the BIN table


## 4. Other native simulation readers

The same dispatcher supports `format="gadget"` for legacy binary snapshots, `format="gadget_hdf5"` for HDF5 snapshots, and `format="gadget-fof"` for legacy Gadget FoF catalogues. Split files are discovered from their shared base path. Position, velocity, and mass conversion factors belong in `reader_params`; see the I/O reference for the complete options.


## 5. Hand the catalogue to `SFCProjection`

The FoF, NPZ, and BIN routes now provide the same objects: positions plus optional catalogue weights and field values. Any of them can be passed directly to an in-memory projection.

```python
task.particle_pos = npz_data["pos"]
task.field_value = npz_data["mass"]
```

Continue to `sfc_projection.ipynb` to build number-, mass-, and redshift-space `SFCField` objects.
